# L3P AE Contrastive Loss Experiment on Kaggle

This notebook compares original L3P against L3P with an auxiliary AE latent triplet loss. The goal is to test whether temporally nearby achieved goals become closer in latent space, and whether that improves long-horizon planning.

How to use on Kaggle:

1. Create a Kaggle Dataset from this repository, or attach a dataset that contains the repo files.
2. Run all cells with `SMOKE = True` first.
3. For a real comparison, set `SMOKE = False`, run matched seeds, commit outputs, then attach previous outputs as input datasets to resume or aggregate.

The baseline command explicitly sets `--ae-contrastive-lambda 0.0`, because the code defaults the new auxiliary loss to `0.1` for the proposed variant.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

def looks_like_repo(root: Path) -> bool:
    return (root / "l3p" / "config.py").exists() and (root / "scripts" / "train_pointmaze.py").exists()

def find_repo() -> Path:
    cwd = Path.cwd()
    if looks_like_repo(cwd):
        return cwd

    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for l3p_dir in base.rglob("l3p"):
            root = l3p_dir.parent
            if looks_like_repo(root):
                return root

    extract_root = Path("/kaggle/working/input_extract")
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for zip_path in input_root.rglob("*.zip"):
            target = extract_root / zip_path.stem
            if not target.exists():
                target.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(zip_path) as zf:
                    zf.extractall(target)
            for l3p_dir in target.rglob("l3p"):
                root = l3p_dir.parent
                if looks_like_repo(root):
                    return root

    raise FileNotFoundError(
        "Could not find the latent_landmarks repo. Attach a Kaggle Dataset containing "
        "the repo, or upload the repo files into this notebook session."
    )

src = find_repo().resolve()
dst = Path("/kaggle/working/latent_landmarks").resolve() if Path("/kaggle/working").exists() else src

if src != dst:
    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns(
            ".git", "logs", "*.pt", "__pycache__", ".pytest_cache", ".ipynb_checkpoints"
        ),
    )

os.chdir(dst)
print("Repo source:", src)
print("Working repo:", Path.cwd())
print("Files ok:", looks_like_repo(Path.cwd()))


In [ ]:
import importlib.util
import subprocess
import sys

required_modules = ["numpy", "torch", "matplotlib", "pandas"]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib", "pandas"
    ])

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("PointMaze is dependency-light; Kaggle CPU is enough for smoke tests.")


## Sanity Check

Run synthetic tests first. They verify AE triples, AE contrastive loss, and the original L3P loss path.

In [ ]:
RUN_TESTS = True

if RUN_TESTS:
    subprocess.run([sys.executable, "tests/test_modules.py"], check=True)


## Experiment Config

`SMOKE = True` checks that the code runs. Do not use smoke results as evidence. For a real comparison, use `SMOKE = False`, several matched seeds, and the same training budget for both variants.

In [ ]:
RUN_TRAINING = True

# True: about 30k env steps, good only for debugging.
# False: use FULL_STEPS below for actual comparison.
SMOKE = True

FULL_STEPS = 500_000
SEEDS = [0]
EVAL_EPISODES = 10 if SMOKE else 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Kaggle runtime safety. In full mode, one baseline+variant pair per session is safer.
CHECKPOINT_EVERY = 5_000 if SMOKE else 50_000
SAVE_TRAINING_STATE = True
RESUME_IF_CHECKPOINT_EXISTS = True
TIME_LIMIT_HOURS_PER_JOB = None if SMOKE else 5.25
RUN_TAG = "smoke" if SMOKE else f"{FULL_STEPS // 1000}k"

# AE contrastive knobs for the proposed variant.
AE_CONTRASTIVE_LAMBDA = 0.1
AE_CONTRASTIVE_MARGIN = 1.0
AE_NEGATIVES_PER_ANCHOR = 1
AE_NEGATIVE_MODE = "random"  # "random" only for now; hard negatives are reserved for later.

OUT_DIR = Path("/kaggle/working/l3p_ae_outputs") if Path("/kaggle/working").exists() else Path("logs/kaggle_ae_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def import_previous_outputs():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return []

    copied = []
    for pattern in ["pm_ae_*_s*.pt", "pm_ae_*_s*.log"]:
        for src_path in input_root.rglob(pattern):
            dst_path = OUT_DIR / src_path.name
            if dst_path.exists():
                continue
            shutil.copy2(src_path, dst_path)
            copied.append(dst_path)
    return copied

previous = import_previous_outputs()

print("mode:", "smoke" if SMOKE else "full")
print("seeds:", SEEDS)
print("run tag:", RUN_TAG)
print("device:", DEVICE)
print("output dir:", OUT_DIR)
print("imported previous artifacts:", len(previous))


In [ ]:
def stream_command(cmd):
    print("\n$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def train_one(variant: str, seed: int):
    assert variant in {"baseline", "ae_contrastive"}
    log_path = OUT_DIR / f"pm_ae_{RUN_TAG}_{variant}_s{seed}.log"
    save_path = OUT_DIR / f"pm_ae_{RUN_TAG}_{variant}_s{seed}.pt"

    cmd = [
        sys.executable,
        "scripts/train_pointmaze.py",
        "--seed", str(seed),
        "--eval-episodes", str(EVAL_EPISODES),
        "--device", DEVICE,
        "--log-file", str(log_path),
        "--save", str(save_path),
        "--save-every", str(CHECKPOINT_EVERY),
    ]

    if SAVE_TRAINING_STATE:
        cmd.append("--save-training-state")
    if RESUME_IF_CHECKPOINT_EXISTS and save_path.exists():
        cmd.extend(["--load", str(save_path)])
    if TIME_LIMIT_HOURS_PER_JOB is not None:
        cmd.extend(["--time-limit-hours", str(TIME_LIMIT_HOURS_PER_JOB)])

    if SMOKE:
        cmd.append("--short")
    else:
        cmd.extend(["--steps", str(FULL_STEPS)])

    if variant == "baseline":
        cmd.extend(["--ae-contrastive-lambda", "0.0"])
    else:
        cmd.extend([
            "--ae-contrastive-lambda", str(AE_CONTRASTIVE_LAMBDA),
            "--ae-contrastive-margin", str(AE_CONTRASTIVE_MARGIN),
            "--ae-negatives-per-anchor", str(AE_NEGATIVES_PER_ANCHOR),
            "--ae-negative-mode", AE_NEGATIVE_MODE,
        ])

    stream_command(cmd)
    return log_path, save_path


## Run Baseline vs AE Contrastive

This runs matched seeds. The only intentional difference is the AE contrastive loss weight and its associated knobs.

In [ ]:
artifacts = []

if RUN_TRAINING:
    for seed in SEEDS:
        artifacts.append(("baseline", seed, *train_one("baseline", seed)))
        artifacts.append(("ae_contrastive", seed, *train_one("ae_contrastive", seed)))

artifacts


## Parse Logs and Plot

Main metric: long-horizon eval success. Diagnostic metrics: `ae_contrastive` and `ae_rank_acc`. The rank metric is the fraction of sampled triples where latent distance already orders positive before negative.

In [ ]:
import re

STEP = re.compile(r"\[\s*(\d+)\s+steps")
EVAL = re.compile(r"eval success rate \(long-horizon test\):\s*([-\d.]+)")
FINAL = re.compile(r"Final long-horizon test success rate:\s*([-\d.]+)")
AE = re.compile(r"ae_contrastive=([-\d.]+)\s+ae_rank_acc=([-\d.]+)")

def parse_log(path: Path):
    stem = path.stem
    variant = "ae_contrastive" if "ae_contrastive" in stem else "baseline"
    seed_match = re.search(r"_s(\d+)", stem)
    seed = int(seed_match.group(1)) if seed_match else -1
    rows = []
    last_step = 0

    with path.open() as f:
        for line in f:
            m = STEP.search(line)
            if m:
                last_step = int(m.group(1))
                ae = AE.search(line)
                if ae:
                    rows.append({
                        "variant": variant,
                        "seed": seed,
                        "step": last_step,
                        "metric": "ae_contrastive",
                        "value": float(ae.group(1)),
                        "log": str(path),
                    })
                    rows.append({
                        "variant": variant,
                        "seed": seed,
                        "step": last_step,
                        "metric": "ae_rank_acc",
                        "value": float(ae.group(2)),
                        "log": str(path),
                    })
                continue

            e = EVAL.search(line)
            if e:
                rows.append({
                    "variant": variant,
                    "seed": seed,
                    "step": last_step,
                    "metric": "eval_success",
                    "value": float(e.group(1)),
                    "log": str(path),
                })
                continue

            final = FINAL.search(line)
            if final:
                rows.append({
                    "variant": variant,
                    "seed": seed,
                    "step": last_step,
                    "metric": "final_success",
                    "value": float(final.group(1)),
                    "log": str(path),
                })
    return rows

rows = []
for log_path in sorted(OUT_DIR.glob("pm_ae_*_s*.log")):
    rows.extend(parse_log(log_path))

df = pd.DataFrame(rows)
display(df.tail(20))

if df.empty:
    print("No rows found yet. Run the training cell first.")
else:
    success = df[df["metric"] == "eval_success"].copy()
    rank = df[df["metric"] == "ae_rank_acc"].copy()
    summary = df[df["metric"].isin(["eval_success", "final_success"])].groupby(
        ["variant", "seed", "metric"]
    )["value"].max().reset_index()
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for variant, color in [("baseline", "tab:blue"), ("ae_contrastive", "tab:orange")]:
        sub = success[success["variant"] == variant]
        if not sub.empty:
            for seed, seed_df in sub.groupby("seed"):
                axes[0].plot(seed_df["step"], seed_df["value"], color=color, alpha=0.25)
            agg = sub.groupby("step")["value"].agg(["mean", "std", "count"]).reset_index()
            axes[0].plot(agg["step"], agg["mean"], marker="o", color=color, label=variant)
            if (agg["count"] > 1).any():
                sd = agg["std"].fillna(0.0)
                axes[0].fill_between(agg["step"], agg["mean"] - sd, agg["mean"] + sd, color=color, alpha=0.15)

        sub_rank = rank[rank["variant"] == variant]
        if not sub_rank.empty:
            agg_rank = sub_rank.groupby("step")["value"].mean().reset_index()
            axes[1].plot(agg_rank["step"], agg_rank["value"], color=color, label=variant)

    axes[0].set_title("Long-horizon eval success")
    axes[0].set_xlabel("Environment steps")
    axes[0].set_ylabel("Success rate")
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].set_title("AE latent ranking accuracy")
    axes[1].set_xlabel("Environment steps")
    axes[1].set_ylabel("P(d_pos < d_neg)")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## Optional Extra Evaluation

Use this after training if you want a cleaner final estimate with more evaluation episodes.

In [ ]:
RUN_EXTRA_EVAL = False
EXTRA_EVAL_EPISODES = 50

if RUN_EXTRA_EVAL:
    for ckpt in sorted(OUT_DIR.glob("pm_ae_*_s*.pt")):
        print("\nEvaluating", ckpt.name)
        stream_command([
            sys.executable,
            "scripts/eval.py",
            "--load", str(ckpt),
            "--episodes", str(EXTRA_EVAL_EPISODES),
        ])


## How to Judge the Result

- Compare matched seeds only: baseline seed 0 vs AE-contrastive seed 0, seed 1 vs seed 1, etc.
- Treat `SMOKE = True` as a wiring test only.
- The main result is long-horizon success rate.
- `ae_rank_acc` should improve or stay high for the contrastive variant; if it does not, the auxiliary loss is not shaping latent geometry as intended.
- If success is worse while rank accuracy improves, try `AE_CONTRASTIVE_LAMBDA = 0.05` before changing other hyperparameters.
- Add hard negatives only after random negatives are stable, so the ablation path remains clean.